# Lab 05-03 · SciPy, SymPy and Matplotlib

Companion to chapter [05-03](../../chapters/05-03-scipy-sympy-matplotlib.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. All random data use fixed seeds, so every number is reproducible.

In [1]:
import math
import numpy as np
import sympy as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq, newton, curve_fit
from scipy.integrate import quad

## Part 1 — Root-finders compared

Solve the diode equation of Example 05-03.1, counting how many times each method evaluates the function.

**🔮 Predict:** which needs the fewest evaluations: bisection, `brentq`, or Newton's method? Which can fail from a poor starting point?

In [2]:
i_s, v_t = 1e-14, 0.025852
calls = {"n": 0}
def mismatch(v):
    calls["n"] += 1
    return (5.0 - v) / 1e3 - i_s * (math.exp(v / v_t) - 1)

def bisection(f, lo, hi, tol=1e-12):
    f_lo = f(lo)
    while hi - lo > tol:
        mid = (lo + hi) / 2
        f_mid = f(mid)
        if f_lo * f_mid <= 0:
            hi = mid
        else:
            lo, f_lo = mid, f_mid
    return (lo + hi) / 2

for name, solve in (("bisection", lambda: bisection(mismatch, 0.0, 1.0)),
                    ("brentq", lambda: brentq(mismatch, 0.0, 1.0, xtol=1e-12)),
                    ("newton from 0.7", lambda: newton(mismatch, 0.7, tol=1e-12))):
    calls["n"] = 0
    root = solve()
    print(f"{name:16} V = {root:.10f} after {calls['n']} evaluations")
for start in (0.0, 0.5, 1.5):
    calls["n"] = 0
    try:
        root = newton(mismatch, start, tol=1e-12, maxiter=50)
        print(f"newton from {start}: returns {root:.4g}, residual {abs(mismatch(root)):.1e} A")
    except RuntimeError:
        print(f"newton from {start}: no convergence in 50 iterations")

bisection        V = 0.6925436332 after 41 evaluations
brentq           V = 0.6925436332 after 14 evaluations
newton from 0.7  V = 0.6925436332 after 7 evaluations
newton from 0.0: returns 5.043e-72, residual 5.0e-03 A
newton from 0.5: returns 0.5, residual 4.5e-03 A
newton from 1.5: no convergence in 50 iterations


Bisection needs 41 evaluations to reach $10^{-12}$; `brentq` 14; SciPy's `newton` (here the secant method, because no derivative was supplied) only 7 from a good guess near 0.7 V. From bad starting points it misbehaves in two ways. From 0 or 0.5 V, where the resistor term dominates and the slope is gentle, the first secant step jumps to 5.0 V or 4.6 V respectively, where the exponential is astronomically large; the next step lands back on the start, two successive estimates agree, and the method's step-size test declares convergence at a point that is *not* a root: the residual is milliamperes, not zero. That silent wrong answer is the worst kind of failure, and the reason to check every root by substituting it back. From 1.5 V, where the exponential dominates, each step moves only about 18 mV (less than one thermal voltage), so it is still creeping toward the root when its 50 iterations run out; allowed 52, it would converge. Slow, not divergent, but from the outside the two look the same. The bracketing methods can do neither: robustness and speed trade off.

## Part 2 — Spotting a wrong model in the residuals

**🔮 Predict:** if you fit a single exponential to data that contain a second, slower exponential, will the fit "look" bad on a plot of data and model?

In [3]:
rng = np.random.default_rng(5)
t = np.linspace(0, 25e-3, 51)
y = 5 * (1 - np.exp(-t / 1e-3)) + 0.5 * (1 - np.exp(-t / 5e-3)) + rng.normal(0, 0.01, t.size)
one = lambda t, vs, tau: vs * (1 - np.exp(-t / tau))
two = lambda t, a1, t1, a2, t2: a1 * (1 - np.exp(-t / t1)) + a2 * (1 - np.exp(-t / t2))
p1, _ = curve_fit(one, t, y, p0=(5, 1e-3))
p2, _ = curve_fit(two, t, y, p0=(5, 1e-3, 0.5, 5e-3))
r1, r2 = y - one(t, *p1), y - two(t, *p2)
print(f"one exponential: rms residual {np.sqrt(np.mean(r1**2))*1e3:.1f} mV")
print(f"two exponentials: rms residual {np.sqrt(np.mean(r2**2))*1e3:.1f} mV, taus {p2[1]*1e3:.3f} and {p2[3]*1e3:.3f} ms")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(t * 1e3, y, ".", ms=4, label="data"); axes[0].plot(t * 1e3, one(t, *p1), label="one-exponential fit")
axes[0].set_xlabel("time (ms)"); axes[0].set_ylabel("voltage (V)"); axes[0].legend()
axes[1].plot(t * 1e3, r1 * 1e3, ".-", label="one exponential"); axes[1].plot(t * 1e3, r2 * 1e3, ".-", label="two exponentials")
axes[1].set_xlabel("time (ms)"); axes[1].set_ylabel("residual (mV)"); axes[1].legend()
fig.tight_layout(); plt.show()

one exponential: rms residual 49.7 mV
two exponentials: rms residual 8.5 mV, taus 1.009 and 5.334 ms


On the left the single exponential looks nearly perfect; on the right its residuals show a smooth, systematic pattern 50 mV in size, while the two-exponential residuals are random at the 10 mV noise level. Always plot residuals.

## Part 3 — Integration accuracy

**🔮 Predict:** the area under $e^t$ from 0 to 1 is exactly $e - 1$. Each time you use four times as many points, by what factor does the error of a left-rectangle sum shrink? Of the trapezoid rule? How many rectangles would match `quad`?

In [4]:
g = math.exp                                      # area from 0 to 1 is exactly e - 1
exact = math.e - 1
for n in (4, 16, 64, 256):
    x = np.linspace(0, 1, n + 1)
    left = float(np.sum([g(v) for v in x[:-1]]) / n)
    trap = float(np.trapezoid([g(v) for v in x], x))
    print(f"n = {n:4d}: left rectangles error {abs(left - exact):.2e}, trapezoid error {abs(trap - exact):.2e}")
val, est = quad(g, 0, 1)
print(f"quad: error {abs(val - exact):.1e} (its own estimate {est:.1e})")

n =    4: left rectangles error 2.06e-01, trapezoid error 8.94e-03
n =   16: left rectangles error 5.31e-02, trapezoid error 5.59e-04
n =   64: left rectangles error 1.34e-02, trapezoid error 3.50e-05
n =  256: left rectangles error 3.35e-03, trapezoid error 2.18e-06
quad: error 2.2e-16 (its own estimate 1.9e-14)


Every fourfold increase in $n$ cuts the rectangle error by 4 (error $\propto 1/n$) and the trapezoid error by 16 (error $\propto 1/n^2$). Extrapolating: the left-rectangle error is about $0.86/n$, so matching `quad`'s $2\times10^{-16}$ would take about $4\times10^{15}$ rectangles; the trapezoid rule (error about $0.14/n^2$) would need about $3\times10^7$ points. `quad` gets there with 21 function evaluations because it uses Gauss–Kronrod rules, which are exact for polynomials of high degree, and places its points adaptively. Its own error estimate ($1.9\times10^{-14}$) is conservative: the actual error is smaller. That is the usual relationship, and the reason to compare the estimate with the result rather than treat it as the error.

## Part 4 — SymPy as a checker

A Wheatstone bridge (two dividers side by side) is balanced, with zero voltage between the dividers' midpoints, when $R_1/R_2 = R_3/R_4$. Check the claim symbolically.

In [5]:
R1, R2, R3, R4, V = sp.symbols("R1 R2 R3 R4 V", positive=True)
v_left = V * R2 / (R1 + R2)
v_right = V * R4 / (R3 + R4)
difference = sp.simplify(v_left - v_right)
print(difference)
print(sp.simplify(difference.subs(R4, R3 * R2 / R1)))

V*(R2*(R3 + R4) - R4*(R1 + R2))/((R1 + R2)*(R3 + R4))


0


The difference vanishes exactly when $R_2R_3 = R_1R_4$, i.e. $R_1/R_2 = R_3/R_4$. Substituting the balance condition gives 0, confirming the hand result. Bridges measure small resistance changes (strain gauges, thermistors) by detecting departures from balance.

## Part 5 — A publication-quality figure

Recreate the chapter's Figure 1 from data and save it as SVG.

In [6]:
fc = 1e3
f = np.logspace(0, 6, 400)
mag = 1 / np.sqrt(1 + (f / fc) ** 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(f / 1e3, mag); axes[0].set_xlabel("frequency (kHz)"); axes[0].set_ylabel("|H| (V/V)")
axes[1].semilogx(f, 20 * np.log10(mag), label="RC low-pass, $f_c$ = 1 kHz")
axes[1].axvline(fc, ls="--", lw=1); axes[1].axhline(-3.01, ls="--", lw=1)
axes[1].set_xlabel("frequency (Hz)"); axes[1].set_ylabel("|H| (dB)"); axes[1].legend()
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig("lowpass-two-ways.svg")
print("saved; dB at fc:", round(float(20 * np.log10(1 / math.sqrt(2))), 3))
plt.show()

saved; dB at fc: -3.01


## Your turn

1. In Part 3, replace $e^t$ with $\sqrt t$ on $[0, 1]$ (exact area 2/3). How do the rectangle and trapezoid errors now shrink as $n$ grows, and why is $\sqrt t$ harder?
2. Now try $\sin^2(\pi t)$ on $[0, 1]$ (exact area ½). What happens, and why is it a special case?
3. In Part 1, start Newton's method at 0.6 V and at 0.9 V. Which converge, and in how many evaluations? Check each answer by substituting it back. Then supply the true derivative (`fprime=`) and try 0.6 V again.

<details><summary>Answers</summary>

1. At $n$ = 4, 16, 64 and 256 the left-rectangle errors are $1.48\times10^{-1}$, $3.43\times10^{-2}$, $8.21\times10^{-3}$ and $2.00\times10^{-3}$, still shrinking roughly as $1/n$; the trapezoid errors are $2.34\times10^{-2}$, $3.09\times10^{-3}$, $3.96\times10^{-4}$ and $5.01\times10^{-5}$, shrinking by about 8 per fourfold step ($n^{-1.5}$) instead of 16. The trapezoid rule's $1/n^2$ error assumes a bounded second derivative; $\sqrt t$ has an infinite slope at $t = 0$, and the first interval, where the chord badly misses the curve, dominates the error. `quad` still reaches $2\times10^{-16}$, because it subdivides adaptively near the trouble spot.
2. Both sums are exact to rounding for every $n \ge 2$. Since $\sin^2(\pi t) = \tfrac12[1 - \cos(2\pi t)]$, the sum of the cosine term over equally spaced points covering whole periods is exactly zero, leaving exactly ½. Smooth periodic functions integrated over whole periods are the one case where equally spaced sums are astonishingly accurate; do not generalize from them (this was the lab's original Part 3, and it hid every difference between the methods).
3. From 0.9 V the secant method converges to 0.6925436332 V in 20 evaluations, with a residual of about $10^{-17}$ A. From 0.6 V it never converges: the iterates cycle 0.6 V → 1.36 V → 0.6 V, because the first secant step overshoots onto the exponential's steep side, and from there the near-vertical secant sends it back. The call raises "Failed to converge" after its iterations run out, which is at least an honest failure. With `fprime=` (true Newton's method), 0.6 V converges after 32 function evaluations (plus as many derivative evaluations): the tangent is a better guide than a secant through two distant points, though it still creeps down the exponential at first. A good starting point, close to the root, matters more than the choice of method.

</details>